# ComicAnalizer - OCR Colab Limpio

Notebook para correr **solo OCR** usando resultados Magi ya procesados.

Orden recomendado:

1. Activar GPU en Colab: `Runtime > Change runtime type > GPU`.
2. Ejecutar instalaci?n y verificaci?n.
3. Subir los 2 ZIPs cuando el notebook lo pida:
   - `magi_clean_full.zip`
   - `colab_clean_full_magi_ocr_outputs.zip`
4. Probar OCR en 1 p?gina.
5. Si funciona, correr OCR completo.

La instalaci?n ocurre antes de subir archivos para no perder tiempo si PaddleOCR falla.

In [ ]:
# 1) Clonar repositorio limpio
!rm -rf /content/ComicAnalizer
!git clone https://github.com/nicolas4432/ComicAnalizer.git /content/ComicAnalizer
%cd /content/ComicAnalizer
!git log --oneline -5

In [ ]:
# 2) Instalar dependencias OCR y verificar PaddleOCR ANTES de subir archivos
%cd /content/ComicAnalizer

import os
import re
import subprocess
import sys

# Variables que reducen crashes nativos en Paddle/PaddleX en Colab.
os.environ["FLAGS_use_mkldnn"] = "0"
os.environ["FLAGS_enable_pir_api"] = "0"
os.environ["PADDLE_PDX_ENABLE_MKLDNN_BYDEFAULT"] = "0"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["PYTHONFAULTHANDLER"] = "1"

print("GPU visible para Colab:")
nvidia = subprocess.run(["nvidia-smi"], text=True, capture_output=True)
print(nvidia.stdout if nvidia.returncode == 0 else "nvidia-smi no disponible")

commands = [
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"],
    [sys.executable, "-m", "pip", "uninstall", "-y", "paddlepaddle", "paddlepaddle-gpu", "paddleocr", "paddlex",
        "torch", "torchvision", "torchaudio"],
    [
        sys.executable, "-m", "pip", "install", "-q",
        "shapely", "pyclipper", "opencv-python-headless", "Pillow", "numpy",
        "langchain", "langchain-core", "langchain-community", "langchain-text-splitters",
    ],
]

# Colab actual normalmente expone CUDA 12.x. Usamos cu126 si hay GPU;
# si no hay GPU, usamos CPU como fallback, pero avisamos claramente.
if nvidia.returncode == 0:
    paddle_install = [
        sys.executable, "-m", "pip", "install", "-q",
        "paddlepaddle-gpu==3.2.2",
        "-i", "https://www.paddlepaddle.org.cn/packages/stable/cu126/",
    ]
else:
    paddle_install = [
        sys.executable, "-m", "pip", "install", "-q",
        "paddlepaddle==3.3.0",
        "-i", "https://www.paddlepaddle.org.cn/packages/stable/cpu/",
    ]

commands.append(paddle_install)
commands.append([sys.executable, "-m", "pip", "install", "-q", "paddleocr==3.5.0"])

for cmd in commands:
    print("Ejecutando:", " ".join(cmd))
    subprocess.run(cmd, check=True)

print("Verificando PaddleOCR...")
import paddle
import paddleocr

print("paddle:", paddle.__version__)
print("paddleocr:", getattr(paddleocr, "__version__", "unknown"))
print("paddle cuda disponible:", paddle.is_compiled_with_cuda())
try:
    print("gpu count:", paddle.device.cuda.device_count())
except Exception as exc:
    print("gpu count no disponible:", exc)

if nvidia.returncode == 0 and not paddle.is_compiled_with_cuda():
    raise RuntimeError(
        "Colab tiene GPU visible, pero Paddle quedo instalado sin CUDA. "
        "Reinicia runtime y vuelve a ejecutar desde la primera celda."
    )

print("PaddleOCR importado correctamente")

## Subir Entradas

Selecciona **los dos ZIPs al mismo tiempo**:

- `magi_clean_full.zip`
- `colab_clean_full_magi_ocr_outputs.zip`

Si Colab renombra alguno como `(1)` o `(2)`, este notebook igual detecta las carpetas reales autom?ticamente.

In [ ]:
# 3) Subir y descomprimir los 2 ZIPs
from google.colab import files
from pathlib import Path
import zipfile
import os

uploaded = files.upload()
uploaded_names = list(uploaded)

print("Subidos:")
for name in uploaded_names:
    print("-", name)

zip_names = [name for name in uploaded_names if name.endswith(".zip")]
if len(zip_names) < 2:
    raise RuntimeError(
        "Debes subir 2 ZIPs a la vez: magi_clean_full.zip y colab_clean_full_magi_ocr_outputs.zip"
    )

dataset_candidates = [
    name for name in zip_names
    if name.startswith("magi_") and "clean" in name
]
if not dataset_candidates:
    raise RuntimeError("No encontre el ZIP del dataset limpio, por ejemplo magi_clean_full.zip")

DATASET_ZIP = dataset_candidates[0]
MAGI_RESULTS_ZIP = next(name for name in zip_names if name != DATASET_ZIP)

print("DATASET_ZIP:", DATASET_ZIP)
print("MAGI_RESULTS_ZIP:", MAGI_RESULTS_ZIP)

!rm -rf /content/magi_sample /content/magi_results_input
!mkdir -p /content/magi_sample /content/magi_results_input

!unzip -q -o "$DATASET_ZIP" -d /content/magi_sample
!unzip -q -o "$MAGI_RESULTS_ZIP" -d /content/magi_results_input

print("Carpetas by_comic encontradas:")
!find /content/magi_sample -maxdepth 6 -type d -name by_comic

print("Archivos Magi encontrados:")
!find /content/magi_results_input -maxdepth 8 \( -name "magi_results.json" -o -name "metrics.json" \) | head -30

In [ ]:
# 4) Configurar rutas automaticamente
from pathlib import Path

RUN_NAME = "colab_clean_full_ocr_full"
SOURCE_RUN_NAME = "colab_clean_full"
DATASET_NAME = "test_1_clean"
COMIC_ID = ""  # "" = todos; ejemplo: "blackbeard-la-leyenda-del-rey-pirata-01"
OCR_SELECTION = "first"  # first, random o suspicious

RUN_ROOT = f"outputs/runs/{RUN_NAME}"
ANALYSIS_OUTPUT = f"{RUN_ROOT}/analysis/magi_analysis_report.json"
OCR_OUTPUT = f"{RUN_ROOT}/analysis/paddle_magi_ocr_comparison.json"
OCR_VISUALS = f"{RUN_ROOT}/visuals/ocr_boxes"
OCR_EVIDENCE_OUTPUT = f"{RUN_ROOT}/analysis/ocr_evidence"

by_comic_candidates = sorted(Path("/content/magi_sample").glob("**/by_comic"))
print("Carpetas by_comic encontradas:")
for candidate in by_comic_candidates:
    print("-", candidate)
if not by_comic_candidates:
    raise RuntimeError("No encontre by_comic dentro de /content/magi_sample")
IMAGE_ROOT = str(by_comic_candidates[0])

preferred_magi = Path(f"/content/magi_results_input/outputs/runs/{SOURCE_RUN_NAME}/magi")
if preferred_magi.exists():
    MAGI_INPUT = str(preferred_magi)
else:
    magi_candidates = sorted(Path("/content/magi_results_input").glob("**/magi"))
    print("Carpetas Magi encontradas:")
    for candidate in magi_candidates:
        print("-", candidate)
    if not magi_candidates:
        raise RuntimeError("No encontre carpeta magi dentro del ZIP de resultados")
    MAGI_INPUT = str(magi_candidates[0])

print("MAGI_INPUT existe:", Path(MAGI_INPUT).exists(), MAGI_INPUT)
print("IMAGE_ROOT existe:", Path(IMAGE_ROOT).exists(), IMAGE_ROOT)
print("OCR_OUTPUT:", OCR_OUTPUT)
print("OCR_VISUALS:", OCR_VISUALS)

In [ ]:
# 5) Analizar resultados Magi existentes, sin reprocesar Magi
!python -m tools.analyze_magi_results \
  --input "$MAGI_INPUT" \
  --image-root "$IMAGE_ROOT" \
  --output "$ANALYSIS_OUTPUT" \
  --visual-output-dir "outputs/runs/colab_clean_full_ocr_full/visuals/magi_boxes" \
  --dataset-name "$DATASET_NAME"

In [ ]:
# 6) Prueba OCR de 1 pagina. Si esta celda funciona, recien corre OCR completo.
import os
import shlex
import subprocess
import sys
from pathlib import Path

TEST_LIMIT = 1

env = os.environ.copy()
env.update({
    "FLAGS_use_mkldnn": "0",
    "FLAGS_enable_pir_api": "0",
    "PADDLE_PDX_ENABLE_MKLDNN_BYDEFAULT": "0",
    "OMP_NUM_THREADS": "1",
    "MKL_NUM_THREADS": "1",
    "OPENBLAS_NUM_THREADS": "1",
    "NUMEXPR_NUM_THREADS": "1",
    "PYTHONFAULTHANDLER": "1",
})

cmd = [
    sys.executable, "-u", "-m", "tools.compare_magi_paddleocr",
    "--magi-input", MAGI_INPUT,
    "--image-root", IMAGE_ROOT,
    "--dataset-name", DATASET_NAME,
    "--selection", OCR_SELECTION,
    "--limit", str(TEST_LIMIT),
    "--seed", "42",
    "--lang", "en",
    "--visual-output-dir", OCR_VISUALS,
    "--output", OCR_OUTPUT,
    "--checkpoint-every", "1",
]
if COMIC_ID:
    cmd.extend(["--comic-id", COMIC_ID])

print("Ejecutando prueba OCR:")
print(" ".join(shlex.quote(part) for part in cmd))
result = subprocess.run(cmd, env=env)
if result.returncode != 0:
    raise RuntimeError(f"La prueba OCR fallo con codigo {result.returncode}")

In [ ]:
# 7) Leer resultado de prueba OCR
import json
from pathlib import Path

ocr_path = Path(OCR_OUTPUT)
partial_path = ocr_path.with_suffix(".partial.json")
print("Existe final:", ocr_path.exists(), ocr_path)
print("Existe partial:", partial_path.exists(), partial_path)

read_path = ocr_path if ocr_path.exists() else partial_path
if not read_path.exists():
    raise RuntimeError("No existe resultado OCR. Ejecuta primero la prueba OCR de 1 pagina.")

ocr_report = json.loads(read_path.read_text(encoding="utf-8"))
print("Leyendo:", read_path)
print(json.dumps(ocr_report["summary"], indent=2, ensure_ascii=False))
for item in ocr_report["comparisons"][:20]:
    print(
        item["comic_id"], item["file_name"],
        "Magi=", item["magi_text_regions"],
        "Paddle=", item["paddle_text_blocks"],
        "match=", item["matched_regions"],
        "t=", round(item["paddle_elapsed_seconds"], 2),
        "err=", item["paddle_error"],
    )

In [ ]:
# 8) OCR completo. Ejecuta esta celda SOLO si la prueba de 1 pagina funciono.
import os
import shlex
import subprocess
import sys

RUN_FULL_OCR = False  # Cambia a True para procesar todas las paginas.

if not RUN_FULL_OCR:
    print("RUN_FULL_OCR esta en False. Cambialo a True cuando quieras procesar todo.")
else:
    env = os.environ.copy()
    env.update({
        "FLAGS_use_mkldnn": "0",
        "FLAGS_enable_pir_api": "0",
        "PADDLE_PDX_ENABLE_MKLDNN_BYDEFAULT": "0",
        "OMP_NUM_THREADS": "1",
        "MKL_NUM_THREADS": "1",
        "OPENBLAS_NUM_THREADS": "1",
        "NUMEXPR_NUM_THREADS": "1",
        "PYTHONFAULTHANDLER": "1",
    })

    cmd = [
        sys.executable, "-u", "-m", "tools.compare_magi_paddleocr",
        "--magi-input", MAGI_INPUT,
        "--image-root", IMAGE_ROOT,
        "--dataset-name", DATASET_NAME,
        "--selection", OCR_SELECTION,
        "--limit", "0",
        "--seed", "42",
        "--lang", "en",
        "--visual-output-dir", OCR_VISUALS,
        "--output", OCR_OUTPUT,
        "--checkpoint-every", "1",
    ]
    if COMIC_ID:
        cmd.extend(["--comic-id", COMIC_ID])

    print("Ejecutando OCR completo:")
    print(" ".join(shlex.quote(part) for part in cmd))
    result = subprocess.run(cmd, env=env)
    if result.returncode != 0:
        raise RuntimeError(f"OCR completo fallo con codigo {result.returncode}")

In [ ]:
# 9) Exportar evidencia OCR para calibracion, si existe OCR final o parcial
from pathlib import Path

ocr_path = Path(OCR_OUTPUT)
partial_path = ocr_path.with_suffix(".partial.json")
read_path = ocr_path if ocr_path.exists() else partial_path

if not read_path.exists():
    print("No hay OCR para exportar evidencia todavia.")
else:
    !python -m tools.export_ocr_evidence \
      --ocr-report "$read_path" \
      --magi-input "$MAGI_INPUT" \
      --image-root "$IMAGE_ROOT" \
      --output-dir "$OCR_EVIDENCE_OUTPUT" \
      --dataset-name "$DATASET_NAME" \
      --asset-policy priority \
      --max-asset-blocks 500


In [ ]:
# 10) Descargar resultados
from google.colab import files
from pathlib import Path

zip_name = "colab_clean_full_ocr_full_outputs.zip"
!zip -qr "$zip_name" outputs/runs/colab_clean_full_ocr_full
print("Descargando:", zip_name)
files.download(zip_name)